# ResNet (Residual Networks)

A refresher on the trick that let us train networks *hundreds* of layers deep: the **residual block** — learn a small correction on top of the identity, `y = F(x) + x`, instead of a fresh transformation from scratch.

**Domain:** Architectures  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

A **ResNet** is a convolutional network built from **residual blocks**. Each block computes `y = F(x) + x`: a couple of conv layers (`F`) whose output is *added back* to the block's input via a **skip / shortcut connection**. Stack dozens or hundreds of these and you get ResNet-18/34/50/101/152 (the number is the layer count).

**The problem it solves — degradation, not overfitting.** By 2015, everyone knew "deeper should be better," yet plain stacks past ~20 layers got *worse* training error — not a generalization gap, the network simply couldn't optimize. This is counter-intuitive: a deeper net can always represent a shallower one by setting the extra layers to the identity. The trouble is that an ordinary stack of nonlinear layers finds it *hard to learn the identity map*. He et al. (2015) reframed the layers to learn a **residual** `F(x) = H(x) − x` instead of the full mapping `H(x)`. If the best thing a block can do is nothing, it just drives `F(x) → 0` — far easier than coaxing a stack of ReLU+conv layers into an exact identity. The skip connection also gives gradients a **direct path** back to early layers, defeating the vanishing-gradient problem.

The payoff was dramatic: ResNet-152 (8× deeper than VGG) won ImageNet 2015 with *lower* complexity, and "add residual connections" became a near-universal default.

**Reach for ResNet when** you need a strong, well-understood CNN backbone: image classification, or as the feature extractor inside detection/segmentation (Faster R-CNN, Mask R-CNN), pose, medical imaging, and as a baseline before anything fancier. Pretrained ImageNet weights for ResNet-18/50 are everywhere and transfer beautifully.

**Don't reach for it when** you have a huge dataset and compute and want the last few points of accuracy — a [`vision-transformer`](vision-transformer.ipynb) or modern ConvNeXt may edge it out — or when the input has no spatial structure. But the *residual connection itself* is not optional these days: Transformers, U-Nets, and almost every deep architecture use it. See [`cnn`](cnn.ipynb) for the convolution basics this builds on.

## 2. Mental Model

**A residual block is a worker who starts by handing the input straight through, then learns only the *edits* worth making.** The shortcut carries `x` forward unchanged; the conv layers `F` learn a *delta* to add. Default behavior is "do nothing" (`F(x)=0`, output = input), and the network spends capacity only where a change actually helps.

```
        x ───────────────┐  (identity shortcut: x passes through untouched)
        │                 │
   ┌────▼────┐            │
   │ conv→BN │            │
   │  →ReLU  │   F(x)     │
   │ conv→BN │            │
   └────┬────┘            │
        │                 ▼
        └──────────────► (+) ──► ReLU ──► y = F(x) + x
```

Two ways to see *why* this helps:

- **Optimization view.** Learning "output ≈ input" is trivial when input is already wired in — just push the weights of `F` toward zero. Without the shortcut, the same identity must be assembled out of nonlinear layers, which is hard, so deep plain nets stall.
- **Gradient-highway view.** Because `y = F(x) + x`, the gradient `∂y/∂x = ∂F/∂x + 1`. That **`+1`** means gradients flow back through the addition *undiminished*, no matter how deep the stack. Errors reach the earliest layers without vanishing. A deep ResNet behaves like an **ensemble of many shorter paths** through the network.

You can read the whole network as `output = x + F1(x) + F2(...) + ...` — an *iterative refinement* of the representation, each block nudging it a little closer to the answer.

## 3. Key Concepts

- **Residual block** — the unit `y = F(x) + x`. `F` is typically `conv→BN→ReLU→conv→BN`; the final ReLU is applied *after* the addition. Two flavors: **BasicBlock** (two 3×3 convs, used in ResNet-18/34) and **Bottleneck** (1×1 → 3×3 → 1×1, used in ResNet-50/101/152 to cut compute).
- **Identity shortcut** — the skip connection when input and output have the *same* shape: literally add `x`. Free, no parameters.
- **Projection shortcut** — when a block changes channel count or downsamples (stride 2), `x` and `F(x)` no longer match. A 1×1 conv (option B in the paper) on the shortcut reshapes `x` so the add works. Used at each stage transition.
- **Bottleneck** — `1×1 (reduce) → 3×3 → 1×1 (restore)`. The 3×3 conv operates on a *narrow* channel dim, so a 50-layer net costs about the same as a 34-layer BasicBlock net.
- **BatchNorm (BN)** — every conv is followed by BN. Essential for ResNet stability; normalizes activations, lets you use higher learning rates. See the BN initialization gotcha below.
- **Stages / downsampling** — the net is grouped into 4 stages; each stage's first block halves spatial size (stride 2) and doubles channels. A global average pool + single linear layer produces the logits.
- **Pre-activation ResNet (v2)** — He et al.'s follow-up reorders to `BN→ReLU→conv` *before* the add, giving an even cleaner gradient path and slightly better very-deep results. Modern default for >100 layers.
- **Depth naming** — ResNet-N counts conv+fc layers: 18, 34 (BasicBlock); 50, 101, 152 (Bottleneck). ResNet-50 (~25M params) is the workhorse.

## 4. Setup

The worked examples below need only **NumPy** — they implement a residual block and a gradient-flow experiment from scratch (CPU, no downloads, runs in a second). The final example shows the idiomatic **PyTorch / torchvision** API for loading a pretrained ResNet; it is gated behind an environment variable so the notebook executes top-to-bottom whether or not Torch is installed.

```bash
pip install numpy            # required for the runnable cells
pip install torch torchvision  # optional: only for the gated PyTorch example
```

In [1]:
import numpy as np

np.set_printoptions(precision=3, suppress=True, linewidth=120)
rng = np.random.default_rng(0)
print("numpy", np.__version__, "— ResNet internals from scratch, CPU-only, no downloads")

numpy 2.4.6 — ResNet internals from scratch, CPU-only, no downloads


## 5. Worked Examples

### Example 1 — A residual block, and why `F(x)+x` makes the identity easy

We implement one residual block (`conv`-style linear layers stand in for convs to keep it tiny) and show the key property: **at initialization a residual block is close to the identity**, and learning to *stay* near identity just means keeping `F` small. The trick is to initialize the block's *last* layer near zero (in real ResNets this is done by zero-initializing the final BatchNorm γ) so the block starts as a clean pass-through.

In [2]:
def relu(x):
    return np.maximum(0, x)

def residual_block(x, W1, W2):
    """y = F(x) + x, with F = W2 @ relu(W1 @ x). Shapes preserved so the skip just adds."""
    F = W2 @ relu(W1 @ x)
    return F + x                      # <- the skip connection

d = 6
x = rng.normal(size=d)

# Random F: the block perturbs its input a lot.
W1 = rng.normal(0, 0.5, (d, d))
W2 = rng.normal(0, 0.5, (d, d))
y_rand = residual_block(x, W1, W2)

# "Zero-init the last layer" (what real ResNets do via BN gamma=0): F(x)=0  =>  block == identity.
W2_zero = np.zeros((d, d))
y_id = residual_block(x, W1, W2_zero)

print("input x                :", x)
print("output (random F)      :", y_rand, " -> block changed the input")
print("output (F last-layer 0):", y_id,   " -> exact identity at init")
print("\nidentity error |y - x| with zero-init:", np.linalg.norm(y_id - x))
print("A plain (non-residual) block can't do this without precisely tuning W to invert relu(W1 x).")

input x                : [ 0.126 -0.132  0.64   0.105 -0.536  0.362]
output (random F)      : [ 1.046  0.207  0.389  0.187 -0.153  0.995]  -> block changed the input
output (F last-layer 0): [ 0.126 -0.132  0.64   0.105 -0.536  0.362]  -> exact identity at init

identity error |y - x| with zero-init: 0.0
A plain (non-residual) block can't do this without precisely tuning W to invert relu(W1 x).


### Example 2 — The gradient highway: residual vs. plain deep stacks

Here is the headline result that motivated ResNet. We build a deep stack of layers two ways — **plain** (`x → layer → layer → …`) and **residual** (`x → x + F(x) → …`) — and measure the gradient magnitude that survives back-propagation to the *first* layer. In the plain net the signal decays geometrically with depth (vanishing gradients); the residual `+x` keeps it alive (real ResNets add BatchNorm to keep that surviving signal well-scaled rather than exploding).

In [3]:
def gradient_to_first_layer(depth, residual, scale=0.8, seed=1):
    """Back-prop a unit gradient through `depth` layers; return its norm at layer 0.

    Each layer: h = relu(W @ h_prev)   (plain)   or   h = h_prev + relu(W @ h_prev)  (residual).
    We track the Jacobian-vector product backward, which is what 'gradient flow' measures.
    """
    r = np.random.default_rng(seed)
    d = 16
    Ws = [r.normal(0, scale / np.sqrt(d), (d, d)) for _ in range(depth)]

    # Forward pass, caching pre-activation masks for relu'(z).
    h = r.normal(size=d)
    masks = []
    for W in Ws:
        z = W @ h
        m = (z > 0).astype(float)
        masks.append(m)
        h = (m * z) + (h if residual else 0)      # residual adds the skip

    # Backward pass: g starts as the gradient at the output, propagate to the input.
    g = np.ones(d)
    for W, m in zip(reversed(Ws), reversed(masks)):
        g_through_F = W.T @ (m * g)               # gradient through relu(W @ .)
        g = g_through_F + (g if residual else 0)  # the '+1' identity path for residual
    return np.linalg.norm(g)

print(f"{'depth':>6} | {'plain grad':>12} | {'residual grad':>14}")
print("-" * 40)
for depth in (5, 10, 20, 40, 80):
    gp = gradient_to_first_layer(depth, residual=False)
    gr = gradient_to_first_layer(depth, residual=True)
    print(f"{depth:>6} | {gp:>12.2e} | {gr:>14.2e}")

print("\nPlain: gradient at layer 0 collapses toward 0 as depth grows -> early layers barely learn.")
print("Residual: the +x path keeps the gradient alive (here it even grows) -> deep nets stay")
print("trainable. Real ResNets add BatchNorm to keep this controlled rather than exploding.")

 depth |   plain grad |  residual grad
----------------------------------------
     5 |     2.58e-01 |       1.51e+01
    10 |     2.92e-02 |       1.22e+02
    20 |     1.02e-04 |       2.22e+03
    40 |     7.16e-12 |       6.84e+05
    80 |     1.05e-24 |       1.83e+11

Plain: gradient at layer 0 collapses toward 0 as depth grows -> early layers barely learn.
Residual: the +x path keeps the gradient alive (here it even grows) -> deep nets stay
trainable. Real ResNets add BatchNorm to keep this controlled rather than exploding.


### Example 3 — A real ResNet with PyTorch / torchvision (gated)

In practice you never hand-roll the convs — `torchvision.models` gives you ResNet-18/50 with pretrained ImageNet weights, and `BasicBlock`/`Bottleneck` are the building blocks above. This cell shows the real API: load a pretrained ResNet-18, count its blocks, and run a forward pass. It executes only if you set `RUN_TORCH=1` and have Torch installed; otherwise it prints the reference snippet so the notebook still runs cleanly.

In [4]:
import os

TORCH_SNIPPET = """
import torch
import torchvision.models as models

# Pretrained ResNet-18 (set weights=None to skip the ~45MB download and use random init).
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
net.eval()

# The 4 stages; each .layer is a stack of BasicBlocks (2 convs + skip each).
for name in ("layer1", "layer2", "layer3", "layer4"):
    stage = getattr(net, name)
    print(name, "->", len(stage), "BasicBlocks,", stage[0].conv1.out_channels, "channels")

# One residual block, unrolled: y = relu(F(x) + identity(x))
block = net.layer1[0]
print("\nBasicBlock:", block)            # conv1->bn1->relu->conv2->bn2 (+ downsample on stage entry)

x = torch.randn(1, 3, 224, 224)           # (batch, RGB, H, W)
with torch.no_grad():
    logits = net(x)
print("\nlogits:", tuple(logits.shape), "(1000 ImageNet classes)")
print("params:", sum(p.numel() for p in net.parameters()) / 1e6, "M")
"""

if os.getenv("RUN_TORCH") == "1":
    try:
        exec(TORCH_SNIPPET)
    except ImportError:
        print("PyTorch/torchvision not installed — run `pip install torch torchvision` first.")
else:
    print("[skipped] set RUN_TORCH=1 (and `pip install torch torchvision`) to run the real ResNet.")
    print("Reference snippet:")
    print(TORCH_SNIPPET)

[skipped] set RUN_TORCH=1 (and `pip install torch torchvision`) to run the real ResNet.
Reference snippet:

import torch
import torchvision.models as models

# Pretrained ResNet-18 (set weights=None to skip the ~45MB download and use random init).
net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
net.eval()

# The 4 stages; each .layer is a stack of BasicBlocks (2 convs + skip each).
for name in ("layer1", "layer2", "layer3", "layer4"):
    stage = getattr(net, name)
    print(name, "->", len(stage), "BasicBlocks,", stage[0].conv1.out_channels, "channels")

# One residual block, unrolled: y = relu(F(x) + identity(x))
block = net.layer1[0]
print("
BasicBlock:", block)            # conv1->bn1->relu->conv2->bn2 (+ downsample on stage entry)

x = torch.randn(1, 3, 224, 224)           # (batch, RGB, H, W)
with torch.no_grad():
    logits = net(x)
print("
logits:", tuple(logits.shape), "(1000 ImageNet classes)")
print("params:", sum(p.numel() for p in net.parameters()) / 1e6, "M

## 6. Gotchas & Pitfalls

- **Shortcut shape mismatch.** You can only `F(x) + x` when shapes match. At every stage transition channels double and spatial size halves, so the shortcut needs a **1×1 conv with stride 2** (a *projection* shortcut) to reshape `x`. Forgetting this is the #1 bug when building a ResNet by hand.
- **ReLU placement.** The final ReLU goes *after* the addition (`relu(F(x)+x)`), not inside `F` before the add — otherwise you clip the skip signal and lose the clean gradient path. (Pre-activation v2 moves BN/ReLU *before* the convs entirely.)
- **Zero-init the last BN's γ.** A standard trick: set the final BatchNorm's scale `γ=0` in each block so the block starts as the identity. This stabilizes early training and reliably adds ~0.5–1% top-1. Many "why won't my deep ResNet train" issues trace back to skipping this.
- **BatchNorm + small batches.** BN statistics get noisy with tiny batches (common in detection/segmentation with big images). Use **GroupNorm** or frozen BN instead, or your accuracy quietly tanks.
- **Forgetting `model.eval()`.** BN and dropout behave differently in train vs. eval mode. Run inference in `train()` mode and BN uses batch stats instead of running averages → garbage predictions on batch size 1.
- **Residual ≠ free depth forever.** Past a point (ResNet-1000+) you still see diminishing returns and overfitting; the connection fixes *optimization*, not *capacity needs*. Use pre-activation and proper regularization for the very deep regime.
- **Mismatched normalization at transfer time.** Pretrained torchvision ResNets expect ImageNet-normalized inputs (mean `[0.485,0.456,0.406]`, std `[0.229,0.224,0.225]`). Feed raw 0–1 images and accuracy drops for no obvious reason.

## 7. When to Use vs Alternatives

| Option | Use it when | Trade-off vs. ResNet |
|---|---|---|
| **ResNet-18/34** | Small/medium datasets, fast baselines, edge/latency budgets | Cheapest; a few points below deeper nets on big data |
| **ResNet-50/101/152** | Standard backbone for classification, detection, segmentation | The reliable workhorse; ResNet-50 is the default to beat |
| **Plain VGG / AlexNet** | Almost never now | No skip connections — harder to train, more params, weaker |
| **DenseNet** | Param efficiency, feature reuse | *Concatenates* instead of adding; more memory-hungry |
| **EfficientNet / ConvNeXt** | Want best accuracy-per-FLOP on a CNN | Beats ResNet on ImageNet but more finicky to train/tune |
| **[Vision Transformer](vision-transformer.ipynb)** | Huge data + compute, want SOTA, global context | Outperforms ResNet at scale; data-hungry, weaker inductive bias on small data |
| **[U-Net](u-net.ipynb)** | Dense prediction (segmentation), often with a ResNet *encoder* | Different task shape; frequently built *on top of* ResNet |

**Bottom line.** ResNet-50 with ImageNet-pretrained weights is the default first thing to try for almost any image task — strong, fast, ubiquitous tooling, and excellent transfer. Reach past it (ViT, ConvNeXt, EfficientNet) only when you have the data/compute and need the last few points. And the residual connection itself is non-negotiable in any deep network you build today.

## 8. Resources

- **Deep Residual Learning for Image Recognition** (He, Zhang, Ren, Sun, 2015) — the original paper, still the clearest explanation: <https://arxiv.org/abs/1512.03385>
- **Identity Mappings in Deep Residual Networks** (He et al., 2016) — the pre-activation "ResNet v2" follow-up: <https://arxiv.org/abs/1603.05027>
- **torchvision ResNet source & weights** — the canonical reference implementation (BasicBlock, Bottleneck, all depths): <https://github.com/pytorch/vision/blob/main/torchvision/models/resnet.py>
- **The Annotated ResNet-50** — a readable, code-walked explainer of the architecture: <https://towardsdatascience.com/the-annotated-resnet-50-a6c536034758>
- **d2l.ai — Residual Networks (ResNet) chapter** — interactive textbook with runnable code: <https://d2l.ai/chapter_convolutional-modern/resnet.html>

Related notebooks: [`cnn`](cnn.ipynb) (the convolutions ResNet stacks), [`vision-transformer`](vision-transformer.ipynb) (the main modern alternative), [`u-net`](u-net.ipynb) (dense-prediction net that often uses a ResNet encoder).